# Bilevel Optimization (Stackelberg games)

A **bilevel** program models a *leader* who optimizes subject to a *follower*
solving its own optimization problem in response:

$$\min_{x,y}\; F(x,y)\quad\text{s.t.}\quad G(x,y)\le 0,\quad
y \in \arg\min_{y'}\{ f(x,y') : g(x,y') \le 0 \}.$$

The leader picks $x$; the follower responds with the $y$ that is optimal for
*its* problem. This models toll setting, network interdiction, pricing, and
Stackelberg competition {cite:p}`Bard1998,Dempe2002,ColsonMarcotteSavard2007`.


## The KKT reduction

When the lower level is **convex in $y$** (for each fixed $x$), the follower's
**KKT conditions** are necessary *and sufficient* for its optimality, so we can
replace `y ∈ argmin{…}` by

- **stationarity** $\nabla_y L = 0$ with $L = f + \sum_i \mu_i g_i$,
- **primal feasibility** $g_i \le 0$,
- **dual feasibility** $\mu_i \ge 0$,
- **complementarity** $0 \le \mu_i \perp -g_i \ge 0$.

The result is a single-level **MPEC**, which discopt already solves
(`discopt.mpec`: GDP / SOS1 / Scholtes). discopt's `BilevelProblem` builds this
reduction automatically. It handles the **optimistic** case; the lower level must
be convex in $y$ — an LP, a convex-QP, or a **convex-NLP** proved convex by an
interval-Hessian test over the box (so the natural linearly-coupled form such as
$e^{y} - x\,y$ is accepted). Nonconvex / integer-follower / pessimistic problems
are refused loudly rather than silently mis-solved
{cite:p}`KleinertLabbeLjubicSchmidt2021`.


### Stationarity needs a *symbolic* derivative

So that $\nabla_y L = 0$ stays inside discopt's certified global path, the
gradient must be an ordinary expression (not an opaque numeric callback). The
`discopt.bilevel.symbolic_diff` engine provides `diff`/`grad` over the
expression DAG:


In [ ]:
import discopt.modeling as dm
from discopt.bilevel import diff

m = dm.Model('demo')
x = m.continuous('x', lb=-5, ub=5)
expr = x**2 + 3 * x
print('d/dx (x^2 + 3x) =', diff(expr, x))   # simplifies to (x + x) + 3


## Worked example: toll setting

A road authority (**leader**) sets a toll $t$ on an arc to maximize revenue
$t\,f$. Drivers (**follower**) choose the flow $f$ to minimize a convex
congestion-plus-toll cost $\tfrac12 f^2 - 5f + t f$. The follower is a convex QP
in $f$, so `BilevelProblem` replaces it by its KKT conditions. We build the
follower's KKT system with `build_kkt_system()` and inspect the emitted
conditions (pure Python — no solver needed for this step):


In [ ]:
from discopt.bilevel import BilevelProblem

m = dm.Model('toll')
t = m.continuous('toll', lb=0, ub=5)   # leader: toll
f = m.continuous('flow', lb=0, ub=8)   # follower: flow
m.minimize(-(t * f))                    # maximize revenue t*f

bl = BilevelProblem(
    m, upper_vars=[t], lower_vars=[f],
    lower_objective=0.5 * f * f - 5 * f + t * f,  # convex QP in f
    lower_constraints=[],                          # only the flow bounds (folded in)
)

# build_kkt_system() emits the follower's KKT conditions (sound math) and returns
# them for inspection, independent of how the complementarity is later encoded.
bl.build_kkt_system()

print('stationarity:', bl.kkt.stationarity[0].body)
for p in bl.kkt.comp_pairs:
    print('complementarity: 0 <=', p.f, 'perp', p.g, '>= 0')


The follower's finite bounds ($0 \le f \le 8$) are folded into the KKT system
automatically — they are genuine follower constraints, so omitting them would be
wrong whenever the follower optimum sits on a bound.

## Solving and the optimum

**Default (uncertified).** Call `bl.formulate(method='strong_duality')` then
`m.solve()`. Strong duality keeps stationarity, primal and dual feasibility, and
replaces the per-row complementarity with the single aggregate equality
$\sum_i \mu_i g_i = 0$ (one bilinear equality, no disjunction). Here the follower's
unconstrained optimum is $f = 5 - t$, so revenue is $t(5-t)$, maximized at
$t^\* = 2.5$, $f^\* = 2.5$ (revenue $6.25$). This is a valid bilevel optimum but is
*not* gap-certified, because the strong-duality equality is itself nonconvex.

**Certified (`gap_certified=True`).** The big-M KKT encodings
(`mpec_method='gdp'`/`'sos1'`) *can* be certified, but only with a **finite bound on
each follower multiplier**: a KKT multiplier has no a-priori upper bound, and a
sentinel-sized big-M is numerically vacuous — it would let the solver certify a
follower-infeasible point (a false optimum), so `BilevelProblem` **refuses the big-M
methods loudly** for unbounded multipliers. Deriving a valid bound automatically is
NP-hard in general {cite:p}`KleinertLabbeLjubicSchmidt2021`, so if you can assert one
for your problem, pass it explicitly:

```python
bl = BilevelProblem(..., multiplier_ub=50.0)   # a *valid* upper bound on the duals
bl.formulate(method='kkt', mpec_method='gdp')
result = m.solve()                              # gap_certified=True
```

The bound is a modeling assertion, exactly like a user-supplied big-M: it must be
$\ge$ the largest multiplier at the true follower optimum for every leader decision,
or it silently cuts the true response. `solve()` warns if a multiplier ends up *at*
its supplied bound.


## References

See the [References](../references.md) page for full citations.